In [1]:
# ── Cell 1: Read from Delta Lake ─────────────────────────────────────────────
CATALOG = "main"
SCHEMA  = "hackathon"
TABLE   = "facilities"
 
df = spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.{TABLE}")
print(f"Row count: {df.count()}")
display(df.limit(5))

NameError: name 'spark' is not defined

In [ ]:
# ── Cell 2: Convert to records and POST to /build-index ──────────────────────
import requests, json
 
records = []
for row in df.collect():
    d = row.asDict()
    # Parse any JSON string columns back to lists
    for col in ["specialties", "capability", "procedure", "equipment",
                "phone_numbers", "affiliation_type_ids"]:
        if isinstance(d.get(col), str):
            try:
                d[col] = json.loads(d[col])
            except Exception:
                d[col] = []
    # Ensure required base fields exist
    d.setdefault("name",     d.get("name", "Unknown"))
    d.setdefault("location", d.get("address_state_or_region", "Unknown"))
    records.append(d)
 
print(f"Prepared {len(records)} records — posting to /build-index...")
resp = requests.post("http://127.0.0.1:8000/build-index", json=records, timeout=300)
print(f"Status: {resp.status_code}")
print(resp.json())

In [ ]:
# ── Cell 3: Test /search ─────────────────────────────────────────────────────
resp = requests.post("http://127.0.0.1:8000/search", json={
    "q":      "Find hospitals with ICU in Northern Ghana",
    "region": "Northern",
    "top_k":  5
})
assert resp.status_code == 200, f"FAIL: /search returned {resp.status_code}"
data = resp.json()
assert data["result_count"] > 0,    "FAIL: /search returned zero facilities"
assert data["confidence"] > 0,      "FAIL: /search confidence is zero"
print(f"PASS /search: {data['result_count']} results, confidence={data['confidence']:.2f}")
display(data["facilities"])

In [ ]:
# ── Cell 4: Test /anomalies ───────────────────────────────────────────────────
resp = requests.get("http://127.0.0.1:8000/anomalies", params={"region": "Northern"})
assert resp.status_code == 200, f"FAIL: /anomalies returned {resp.status_code}"
data = resp.json()
print(f"PASS /anomalies: {data['total_flags']} flag(s) found in Northern region")
for flag in data["flags"]:
    print(f"  ⚠ {flag['facility_name']} — {flag['flag_type']} (confidence: {flag['confidence']:.0%})")

In [ ]:
# ── Cell 5: Test /deserts ─────────────────────────────────────────────────────
for specialty in ["cardiology", "pediatrics", "ophthalmology"]:
    resp = requests.get("http://127.0.0.1:8000/deserts", params={"specialty": specialty})
    assert resp.status_code == 200, f"FAIL: /deserts returned {resp.status_code}"
    data = resp.json()
    print(f"PASS /deserts ({specialty}): {data['total_zones']} desert zone(s)")
    for zone in data["zones"]:
        print(f"  🔴 {zone['region']} — {zone['facility_count']} {specialty} facility (severity: {zone['severity']})")

In [ ]:
# ── Cell 6: Test /coverage (used by frontend heatmap) ────────────────────────
resp = requests.get("http://127.0.0.1:8000/coverage")
assert resp.status_code == 200, f"FAIL: /coverage returned {resp.status_code}"
data = resp.json()
print(f"PASS /coverage: {data['total_regions']} regions in map")
# Show regions with zero cardiology coverage
accra = data["regions"].get("Greater Accra", {})
print(f"  Greater Accra cardiology coverage: {accra.get('cardiology', 0)} facilities")

In [ ]:
# ── Cell 7: MLflow prompt comparison experiment ───────────────────────────────
# Run on a sample of 5 real Ghana records
from backend.idp.extraction import MedicalIDP
 
engine = MedicalIDP()
 
# Use 5 of the real facility description texts from the Delta Lake table
test_texts = (
    spark.sql(f"""
        SELECT description
        FROM {CATALOG}.{SCHEMA}.{TABLE}
        WHERE description IS NOT NULL
          AND length(description) > 50
        LIMIT 5
    """)
    .collect()
)
test_texts = [row["description"] for row in test_texts]
 
if test_texts:
    results = engine.run_prompt_comparison(test_texts)
    print(f"\nPrompt comparison results:")
    print(f"  Zero-shot mean fields: {results['zero_shot_mean']:.1f}")
    print(f"  Few-shot  mean fields: {results['few_shot_mean']:.1f}")
    print(f"  Winner: {results['winner']}")
    print(f"  View in MLflow: /Shared/IDP-Backend-API")
else:
    print("No description texts found in Delta table — add description column or use test texts")


In [ ]:
# ── Cell 8: Final validation summary ─────────────────────────────────────────
print("\n" + "="*60)
print("DAY 2+3 BACKEND VALIDATION COMPLETE")
print("="*60)
checks = {
    "/health":      requests.get("http://127.0.0.1:8000/health").status_code == 200,
    "/extract":     requests.post("http://127.0.0.1:8000/extract",
                        json={"text": "The Korle-Bu Teaching Hospital in Accra has 2000 beds, "
                                      "487 doctors, and a Level I Trauma Center with ICU."}).status_code == 200,
    "/search":      requests.post("http://127.0.0.1:8000/search",
                        json={"q": "ICU hospitals"}).status_code == 200,
    "/anomalies":   requests.get("http://127.0.0.1:8000/anomalies").status_code == 200,
    "/deserts":     requests.get("http://127.0.0.1:8000/deserts",
                        params={"specialty": "cardiology"}).status_code == 200,
    "/coverage":    requests.get("http://127.0.0.1:8000/coverage").status_code == 200,
    "/facilities":  requests.get("http://127.0.0.1:8000/facilities").status_code == 200,
}
 
all_pass = True
for endpoint, passed in checks.items():
    status = "PASS ✓" if passed else "FAIL ✗"
    print(f"  {status}  {endpoint}")
    if not passed:
        all_pass = False
 
print(f"\nOverall: {'ALL ENDPOINTS PASSING ✓' if all_pass else 'SOME ENDPOINTS FAILING — check above'}")
print("MLflow experiment: /Shared/IDP-Backend-API")